# 语义分块的原理与边界

语义分块通常先把文本切成句子，再计算相邻句子或句组的 Embedding 距离。当距离突然增大时，将该位置视为主题边界。

旧教程常使用 `langchain_experimental.text_splitter.SemanticChunker`，但 `langchain-experimental` 已停止维护，且该类没有迁移到稳定的 `langchain-text-splitters`。本节只学习算法思想，并写一个最小教学实现。

## 最小流程

```text
文本 → 句子列表 → 句子 Embedding → 相邻余弦距离
     → 确定阈值 → 在距离突变处断开 → Chunk
```

下面使用项目已有的 Qwen Embedding 服务。该实现用于理解原理，不包含生产环境需要的句组窗口、最小块合并、最大长度兜底和批处理优化。

In [ ]:
import math
import os
import re
from statistics import mean, pstdev

from dotenv import load_dotenv
from langchain.embeddings import init_embeddings

load_dotenv(override=True)
model_name = os.getenv("EMBEDDING_MODEL")
embedding_model = init_embeddings(
    model=f"openai:{model_name}",
    api_key=os.getenv("EMBEDDING_API_KEY"),
    base_url=os.getenv("EMBEDDING_API_URL"),
    dimensions=1024,
)

def cosine_distance(left: list[float], right: list[float]) -> float:
    dot = sum(a * b for a, b in zip(left, right, strict=True))
    left_norm = math.sqrt(sum(value * value for value in left))
    right_norm = math.sqrt(sum(value * value for value in right))
    return 1 - dot / (left_norm * right_norm)


In [ ]:
text = (
    "Python 是一种易读的编程语言。它广泛用于后端和数据分析。"
    "异步编程可以提高 I/O 密集任务的吞吐量。"
    "故宫位于北京。它保存了大量古代建筑和文物。"
    "游客通常从午门进入参观。"
)
sentences = [
    sentence.strip()
    for sentence in re.split(r"(?<=[。！？!?])", text)
    if sentence.strip()
]
vectors = embedding_model.embed_documents(sentences)
distances = [
    cosine_distance(vectors[index], vectors[index + 1])
    for index in range(len(vectors) - 1)
]
threshold = mean(distances) + 0.5 * pstdev(distances)

chunks = []
current = [sentences[0]]
for index, distance in enumerate(distances):
    if distance > threshold:
        chunks.append("".join(current))
        current = []
    current.append(sentences[index + 1])
chunks.append("".join(current))

print("distances=", distances)
print("threshold=", threshold)
for index, chunk in enumerate(chunks, start=1):
    print(index, chunk)


## 边界

语义分块需要额外 Embedding 调用，阈值对数据集敏感，产生的块长度也不稳定。它不天然优于结构切分；当文档已有可靠标题、段落和表格结构时，应优先利用结构。是否值得使用必须通过 Recall、答案正确率、成本和延迟评估。